# Lab Day 1 — Xây dựng mạng nơ-ron và thí nghiệm huấn luyện

**Sinh viên:** 2A202603007
**Bài toán:** Phân loại 7 lớp Forest CoverType (Blackard & Dean, UCI)
**Mục tiêu:** Xây dựng MLP từ đầu bằng PyTorch, thực nghiệm khảo sát 7 yếu tố ảnh hưởng đến quá trình huấn luyện, thiết lập baseline với đo lường độ nhiễu seed, và đánh giá mô hình tối ưu trên tập eval độc lập.


In [1]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch

REPO_ROOT = "../.."     # Thư mục gốc repo
OUT_DIR   = ".."        # Thư mục nộp bài submission_2A202603007/

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")
else:
    print("Using CPU")

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


PyTorch version: 2.8.0+cu129
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
BF16 supported: True


## Part 0 — Chuẩn bị dữ liệu và Tiền xử lý

Dữ liệu CoverType gồm 581 012 mẫu, 54 đặc trưng (10 cột số liên tục và 44 cột nhị phân one-hot).
Tập train (464 809 mẫu) được tách 20% làm validation (92 962 mẫu) theo phương pháp phân tầng (stratified, seed 42).
Thống kê chuẩn hoá (mean/std của 10 cột liên tục) được tính **hoàn toàn chỉ trên tập train còn lại (371 847 mẫu)** để ngăn ngừa rò rỉ thông tin.


In [2]:
# Chuẩn bị dữ liệu và chuyển lên GPU
data = prepare_data(device=device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")

assert data["X_tr"].shape == (371847, 54)
assert data["X_val"].shape == (92962, 54)
assert data["X_eval"].shape == (116203, 54)
print("Xác nhận kích thước dữ liệu hoàn toàn chính xác!")


Dataset summary (on device: cuda):
  Train: X torch.Size([371847, 54]), y torch.Size([371847])
  Val  : X torch.Size([92962, 54]), y torch.Size([92962])
  Eval : X torch.Size([116203, 54]), y torch.Size([116203])
  Majority class on Val: class 1 with accuracy 0.4876 (~0.4876)
  Train numeric mean max abs: 0.00032 (≈ 0)
  Train numeric std max diff vs 1: 0.00011 (≈ 0)
Xác nhận kích thước dữ liệu hoàn toàn chính xác!


## Part 1 — Định nghĩa Model và Kiểm tra Sức khoẻ Ban đầu

Quy định kiến trúc:
- `M-base`: 54 → 256 → 128 → 7 (47 879 tham số)
- `M-wide`: 54 → 512 → 256 → 7 (161 287 tham số)
- `M-deep`: 54 → 256 → 128 → 64 → 7 (55 687 tham số)

Thực hiện các phép thử bắt buộc của slide Chương 5 trước khi huấn luyện dài.


In [3]:
import math

# 1. Khởi tạo mô hình và assert số tham số
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_p = count_params(model)
print(f"Số tham số M-base: {n_p} (kỳ vọng: {EXPECTED_PARAMS[(256, 128)]})")
assert n_p == EXPECTED_PARAMS[(256, 128)]

# 2. Kiểm tra shape forward pass
dummy_batch = torch.randn(8, 54, device=device)
logits = model(dummy_batch)
print(f"Logits shape: {logits.shape} (kỳ vọng: torch.Size([8, 7]))")
assert logits.shape == (8, 7)

# 3. Kiểm tra Step 0 Loss
model.eval()
step0_eval = evaluate(model, data["X_val"], data["y_val"], loss_name="ce")
ln7 = math.log(7)
print(f"Loss bước 0 trên Val: {step0_eval['loss']:.4f} (ln 7 = {ln7:.4f}, sai lệch: {abs(step0_eval['loss'] - ln7):.4f})")
assert abs(step0_eval["loss"] - ln7) < 0.1

# 4. Quá khớp 20 mẫu (Overfit small batch)
m_overfit = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
m_overfit.train()
opt_overfit = torch.optim.Adam(m_overfit.parameters(), lr=0.01)
x_20 = data["X_tr"][:20]
y_20 = data["y_tr"][:20]
for step in range(300):
    opt_overfit.zero_grad()
    loss = torch.nn.functional.cross_entropy(m_overfit(x_20), y_20)
    loss.backward()
    opt_overfit.step()

m_overfit.eval()
with torch.no_grad():
    pred_20 = torch.argmax(m_overfit(x_20), dim=1)
    acc_20 = (pred_20 == y_20).float().mean().item()
    loss_20 = torch.nn.functional.cross_entropy(m_overfit(x_20), y_20).item()
print(f"Quá khớp 20 mẫu sau 300 bước: Loss = {loss_20:.6f}, Accuracy = {acc_20*100:.1f}%")
assert acc_20 == 1.0 and loss_20 < 0.05

# 5. Kiểm tra dòng chảy gradient (Gradient Flow)
model.train()
model.zero_grad()
loss_flow = torch.nn.functional.cross_entropy(model(data["X_tr"][:64]), data["y_tr"][:64])
loss_flow.backward()
print("Kiểm tra chuẩn gradient từng tham số:")
for name, param in model.named_parameters():
    assert param.grad is not None and param.grad.norm().item() > 0
    print(f"  {name:15s}: grad_norm = {param.grad.norm().item():.6f}")


Số tham số M-base: 47879 (kỳ vọng: 47879)
Logits shape: torch.Size([8, 7]) (kỳ vọng: torch.Size([8, 7]))
Loss bước 0 trên Val: 1.9635 (ln 7 = 1.9459, sai lệch: 0.0176)
Quá khớp 20 mẫu sau 300 bước: Loss = 0.000002, Accuracy = 100.0%
Kiểm tra chuẩn gradient từng tham số:
  net.0.weight   : grad_norm = 0.555197
  net.0.bias     : grad_norm = 0.256006
  net.2.weight   : grad_norm = 1.813399
  net.2.bias     : grad_norm = 0.335712
  net.4.weight   : grad_norm = 1.916736
  net.4.bias     : grad_norm = 0.481559


**Nhận xét Part 1:**
1. **Loss bước 0**: Đo được $1.9635$, sát với giá trị lý thuyết $\ln 7 \approx 1.9459$ (chênh lệch chỉ $0.0176$). Điều này khẳng định khởi tạo He đối xứng quanh 0, các điểm số logit đầu ra gần 0 khiến phân phối xác suất dự đoán đồng đều $1/7$ cho mỗi lớp.
2. **Quá khớp 20 mẫu**: Đạt độ chính xác tuyệt đối $100.0\%$ với loss rơi về $2 \times 10^{-6}$, chứng minh mô hình có năng lực biểu diễn tốt, không bị lỗi gán nhãn, không tính softmax hai lần hay quên `zero_grad`.
3. **Gradient Flow**: Chuẩn gradient của mọi lớp (cả weight và bias) đều khác `None` và mang giá trị dương rõ rệt, chứng minh tín hiệu gradient được lan truyền ngược đầy đủ qua toàn bộ các tầng mạng.


## Part 2 — Pipeline Huấn Luyện và Baseline (Đo độ nhiễu 3 seed)

Cấu hình Baseline:
- Model: `M-base` (54 → 256 → 128 → 7, 47 879 tham số)
- Khởi tạo: He normal, bias = 0
- Mất mát: Cross-Entropy
- Optimizer: SGD + momentum 0.9, $lr=0.05$
- Batch size: 512, Epochs: 20, Precision: FP32
- Chạy với 3 seed độc lập ($1, 2, 3$) để xác định ngưỡng nhiễu thống kê $2\sigma$.


In [4]:
# Đọc kết quả 3 seed baseline đã lưu
base_runs = [load_results(f"{OUT_DIR}/results")[i] for i, r in enumerate(load_results(f"{OUT_DIR}/results")) if r["cfg"]["exp_id"] in ["base-s1", "base-s2", "base-s3"]]

accs = [r["summary"]["val_acc"] for r in base_runs]
f1s = [r["summary"]["val_macro_f1"] for r in base_runs]

mean_acc, std_acc = np.mean(accs), np.std(accs, ddof=1)
mean_f1, std_f1 = np.mean(f1s), np.std(f1s, ddof=1)
noise_2sigma = 2 * std_f1

print(f"Baseline Val Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
print(f"Baseline Val Macro-F1: {mean_f1:.4f} ± {std_f1:.4f}")
print(f"Ngưỡng nhiễu 2σ (Val Macro-F1): {noise_2sigma:.4f}")

for r in base_runs:
    s = r["summary"]
    print(f"  {r['cfg']['exp_id']}: Val Acc = {s['val_acc']:.4f} | Val F1 = {s['val_macro_f1']:.4f} | Best Epoch = {s['best_epoch']}")


Baseline Val Accuracy: 0.9016 ± 0.0011
Baseline Val Macro-F1: 0.8422 ± 0.0063
Ngưỡng nhiễu 2σ (Val Macro-F1): 0.0126
  base-s1: Val Acc = 0.9011 | Val F1 = 0.8399 | Best Epoch = 19
  base-s2: Val Acc = 0.9029 | Val F1 = 0.8374 | Best Epoch = 20
  base-s3: Val Acc = 0.9008 | Val F1 = 0.8493 | Best Epoch = 20


**Nhận xét Baseline:**
- Mô hình baseline hội tụ ổn định, vượt xa mốc "đoán lớp đa số" (accuracy $0.4876$, macro-F1 $\approx 0.094$).
- Độ lệch chuẩn giữa 3 seed rất nhỏ: $\sigma_{F1} = 0.0063$, cho thấy tính lặp lại cao.
- **Ngưỡng nhiễu quy ước:** $2\sigma = 0.0126$. Bất kỳ cải tiến nào có $\Delta \text{Macro-F1} > 0.0126$ mới được xem là có ý nghĩa thống kê vượt trội hơn baseline.


## Part 3 — Thí Nghiệm Khảo Sát 7 Chủ Đề Huấn Luyện

Thực hiện đầy đủ 7 chủ đề: Loss, Optimizer, Hyper-parameters, Dropout, Gradient Clipping, Mixed Precision, Weight Initialization.
Mỗi thí nghiệm tuân thủ nguyên tắc: chỉ đổi đúng một yếu tố so với baseline, có dự đoán trước và đối chiếu cơ chế sau khi chạy.


### Chủ đề 1 — Hàm Mất Mát: Cross-Entropy vs. MSE
- **Dự đoán trước:** Cross-Entropy sẽ vượt trội hơn MSE rất nhiều. Với phân loại 7 lớp, đạo hàm của CE là $(p_c - y_c)$ không bị bão hoà khi dự đoán sai lệch lớn. Ngược lại, MSE trên softmax/one-hot có đạo hàm tỉ lệ với $p_c(1-p_c)$, khiến gradient bị suy hao (vanishing gradient) khi mô hình tự tin sai.


In [5]:
ce_res = load_res("loss-ce")["summary"]
mse_res = load_res("loss-mse")["summary"]
print(f"Cross-Entropy: Val Acc = {ce_res['val_acc']:.4f}, Val Macro-F1 = {ce_res['val_macro_f1']:.4f}")
print(f"MSE Loss     : Val Acc = {mse_res['val_acc']:.4f}, Val Macro-F1 = {mse_res['val_macro_f1']:.4f}")
print(f"Chênh lệch Macro-F1 (CE - MSE): +{ce_res['val_macro_f1'] - mse_res['val_macro_f1']:.4f} (Vượt xa 2σ = {noise_2sigma:.4f})")


Cross-Entropy: Val Acc = 0.9011, Val Macro-F1 = 0.8399
MSE Loss     : Val Acc = 0.8544, Val Macro-F1 = 0.6988
Chênh lệch Macro-F1 (CE - MSE): +0.1411 (Vượt xa 2σ = 0.0126)


**Đối chiếu & Cơ chế:** Đúng như dự đoán, CE cho Macro-F1 cao hơn MSE tới $0.1411$ ($83.99\%$ so với $69.88\%$). Vì nhãn mất cân bằng nặng, các lớp thiểu số chịu ảnh hưởng nặng nhất từ hiện tượng gradient suy hao của MSE.


### Chủ đề 2 — Bộ Tối Ưu Hoá: SGD, SGD+Momentum, Adam, AdamW
- **Dự đoán trước:** Adam và AdamW sẽ hội tụ nhanh hơn và đạt kết quả tốt hơn SGD nhờ khả năng thích nghi tốc độ học riêng biệt cho từng tham số (dựa trên moment bậc 1 và bậc 2). Khi $weight\_decay=0$, AdamW sẽ cho kết quả giống hệt Adam.


In [6]:
opt_ids = [
    "opt-sgd-lr0.05", "opt-sgd-lr0.2",
    "opt-sgdm-lr0.01", "opt-sgdm-lr0.05", "opt-sgdm-lr0.1",
    "opt-adam-lr3e-4", "opt-adam-lr1e-3", "opt-adam-lr3e-3",
    "opt-adamw-lr1e-3", "opt-adamw-lr3e-3", "opt-adamw-wd0"
]
print(f"{'Exp ID':20s} | {'Optimizer':14s} | {'LR':7s} | {'WD':5s} | {'Val Acc':8s} | {'Val F1':8s} | {'Best Ep':7s}")
print("-" * 80)
for eid in opt_ids:
    r = load_res(eid)
    cfg, s = r["cfg"], r["summary"]
    print(f"{eid:20s} | {cfg['optimizer']:14s} | {str(cfg['lr']):7s} | {str(cfg['weight_decay']):5s} | {s['val_acc']:.4f}   | {s['val_macro_f1']:.4f}   | {s['best_epoch']:^7d}")


Exp ID               | Optimizer      | LR      | WD    | Val Acc  | Val F1   | Best Ep
--------------------------------------------------------------------------------
opt-sgd-lr0.05       | sgd            | 0.05    | 0.0   | 0.8333   | 0.6925   |    19  
opt-sgd-lr0.2        | sgd            | 0.2     | 0.0   | 0.8709   | 0.7904   |    18  
opt-sgdm-lr0.01      | sgd_momentum   | 0.01    | 0.0   | 0.8688   | 0.7643   |    20  
opt-sgdm-lr0.05      | sgd_momentum   | 0.05    | 0.0   | 0.9011   | 0.8399   |    19  
opt-sgdm-lr0.1       | sgd_momentum   | 0.1     | 0.0   | 0.9070   | 0.8411   |    20  
opt-adam-lr3e-4      | adam           | 0.0003  | 0.0   | 0.8713   | 0.7950   |    20  
opt-adam-lr1e-3      | adam           | 0.001   | 0.0   | 0.9042   | 0.8512   |    19  
opt-adam-lr3e-3      | adam           | 0.003   | 0.0   | 0.9162   | 0.8709   |    19  
opt-adamw-lr1e-3     | adamw          | 0.001   | 0.01  | 0.9038   | 0.8491   |    19  
opt-adamw-lr3e-3     | adamw          |

**Đối chiếu & Cơ chế:**
1. **So sánh công bằng ở lr tối ưu:**
   - SGD ($lr=0.2$): Macro-F1 = $0.7904$
   - SGDM ($lr=0.1$): Macro-F1 = $0.8411$
   - Adam ($lr=0.003$): Macro-F1 = $0.8709$
   - AdamW ($lr=0.003$): Macro-F1 = $0.8746$
   AdamW giành chiến thắng, vượt baseline $+0.0347$ (vượt xa $2\sigma = 0.0126$).
2. **Kiểm chứng lý thuyết AdamW với $wd=0$:** Kết quả của `opt-adamw-wd0` giống hệt `opt-adam-lr1e-3` (cùng Val Acc $0.9042$, Val F1 $0.8512$, Best Epoch $19$), chứng minh khi không có suy giảm trọng số thì AdamW và Adam hoàn toàn đồng nhất.


### Chủ đề 3 — Hyper-parameters: Batch Size, Độ Rộng, Độ Sâu
- **Dự đoán trước:** Batch size nhỏ hơn (128) sẽ thực hiện nhiều bước cập nhật hơn trong cùng 20 epoch nên sẽ học nhanh hơn. Kiến trúc lớn hơn (`M-wide`, `M-deep`) sẽ tăng năng lực biểu diễn và cải thiện macro-F1.


In [7]:
hp_ids = ["hp-batch-128", "hp-batch-512", "hp-batch-2048", "hp-arch-wide", "hp-arch-deep", "hp-wd-1e-4"]
print(f"{'Exp ID':16s} | {'Yếu tố khảo sát':25s} | {'Val Acc':8s} | {'Val F1':8s} | {'Best Ep':7s}")
print("-" * 75)
for eid in hp_ids:
    r = load_res(eid)
    print(f"{eid:16s} | {r['cfg']['description'][:25]:25s} | {r['summary']['val_acc']:.4f}   | {r['summary']['val_macro_f1']:.4f}   | {r['summary']['best_epoch']:^7d}")


Exp ID           | Yếu tố khảo sát           | Val Acc  | Val F1   | Best Ep
---------------------------------------------------------------------------
hp-batch-128     | Batch nhỏ 128 (nhiều bướ  | 0.9154   | 0.8675   |    20  
hp-batch-512     | Batch chuẩn 512           | 0.9011   | 0.8399   |    19  
hp-batch-2048    | Batch lớn 2048 (ít bước   | 0.8732   | 0.7757   |    20  
hp-arch-wide     | M-wide (512 -> 256 -> 7,  | 0.9146   | 0.8668   |    20  
hp-arch-deep     | M-deep (256 -> 128 -> 64  | 0.9182   | 0.8661   |    19  
hp-wd-1e-4       | SGD+mom với L2 regulariz  | 0.8931   | 0.8291   |    19  


**Đối chiếu & Cơ chế:**
- **Batch Size:** Batch 128 có 2905 bước/epoch đạt F1 = $0.8675$, trong khi Batch 2048 chỉ có 182 bước/epoch chỉ đạt F1 = $0.7757$.
- **Kiến trúc:** Cả `M-wide` (161k params) và `M-deep` (55k params) đều cải thiện Macro-F1 lên $\approx 0.866$ (+0.027 so với M-base), chứng minh việc tăng chiều không gian ẩn hoặc độ sâu giúp mô hình xẻ ranh giới quyết định phi tuyến tốt hơn.


### Chủ đề 4 — Dropout ($q = 0.0, 0.2, 0.4$)
- **Dự đoán trước:** Vì tập dữ liệu rất lớn (371 847 mẫu train) và mạng M-base có kích thước vừa phải (47k params), mô hình **chưa hề bị quá khớp nặng**. Do đó, thêm dropout sẽ làm giảm độ chính xác vì làm suy giảm năng lực biểu diễn hiệu dụng.


In [8]:
drop_ids = ["drop-0.0", "drop-0.2", "drop-0.4"]
for eid in drop_ids:
    r = load_res(eid)
    cfg, s = r["cfg"], r["summary"]
    print(f"Dropout q={cfg['dropout']}: Val Acc = {s['val_acc']:.4f} | Val Macro-F1 = {s['val_macro_f1']:.4f} | Final Train Loss = {s['final_train_loss']:.4f} | Final Val Loss = {s['final_val_loss']:.4f}")


Dropout q=0.0: Val Acc = 0.9011 | Val Macro-F1 = 0.8399 | Final Train Loss = 0.2312 | Final Val Loss = 0.2398
Dropout q=0.2: Val Acc = 0.8770 | Val Macro-F1 = 0.7955 | Final Train Loss = 0.2974 | Final Val Loss = 0.3012
Dropout q=0.4: Val Acc = 0.8446 | Val Macro-F1 = 0.7052 | Final Train Loss = 0.3851 | Final Val Loss = 0.3887


**Đối chiếu & Cơ chế:**
Khoảng cách giữa Train Loss và Val Loss ở $q=0.0$ chỉ là $0.2398 - 0.2312 = 0.0086$ (hầu như không quá khớp!). Do đó, áp dụng dropout $q=0.2$ hoặc $q=0.4$ gây phản tác dụng: Macro-F1 tụt dốc từ $0.8399 \rightarrow 0.7955 \rightarrow 0.7052$.
**Kết luận:** Dropout là phương thuốc trị quá khớp, không nên dùng khi mô hình còn đang underfitting hoặc dữ liệu đủ lớn.


### Chủ đề 5 — Cắt Gradient (Gradient Clipping)
- **Dự đoán trước:** Ở lr bình thường ($0.05$), gradient hiếm khi đột biến nên clipping $c=1.0$ không tạo ra khác biệt lớn. Ở lr rất cao ($lr=2.0$), không clip sẽ làm gradient bùng nổ phá vỡ trọng số; ngược lại có clip sẽ kiềm chế bước nhảy và bảo vệ mạng khỏi sụp đổ.


In [9]:
clip_ids = ["clip-none-normlr", "clip-1.0-normlr", "clip-none-highlr", "clip-1.0-highlr"]
for eid in clip_ids:
    r = load_res(eid)
    cfg, s = r["cfg"], r["summary"]
    print(f"{eid:18s} | lr={cfg['lr']} | clip={str(cfg['clip_norm']):4s} | Val Acc = {s['val_acc']:.4f} | Val F1 = {s['val_macro_f1']:.4f} | Best Ep = {s['best_epoch']}")


clip-none-normlr   | lr=0.05 | clip=None | Val Acc = 0.9011 | Val F1 = 0.8399 | Best Ep = 19
clip-1.0-normlr    | lr=0.05 | clip=1.0  | Val Acc = 0.8977 | Val F1 = 0.8311 | Best Ep = 19
clip-none-highlr   | lr=2.0  | clip=None | Val Acc = 0.4876 | Val F1 = 0.0936 | Best Ep = 15
clip-1.0-highlr    | lr=2.0  | clip=1.0  | Val Acc = 0.7021 | Val F1 = 0.3101 | Best Ep = 1


**Đối chiếu & Cơ chế:**
- Ở $lr=0.05$: Gradient norm trung bình $\approx 0.3 - 0.4$ nên clipping $c=1.0$ hầu như không tác động.
- Ở $lr=2.0$: Không có clipping khiến gradient bùng nổ, mô hình hỏng hoàn toàn (Val Acc rơi về đúng $0.4876$, Val F1 $= 0.0936$ tương đương đoán mò). Khi có clipping $c=1.0$, bước cập nhật được chuẩn hoá, giữ mạng không bị sụp đổ hoàn toàn.


### Chủ đề 6 — Mixed Precision: FP32 vs. FP16 vs. BF16
- **Dự đoán trước:** Độ chính xác của FP16 và BF16 sẽ tương đương FP32. Tuy nhiên, thời gian huấn luyện trên M-base có thể không nhanh hơn đáng kể vì kích thước mạng nhỏ (47k params), chi phí kernel launch và type casting chiếm tỉ trọng lớn so với thời gian tính toán Tensor Core.


In [10]:
amp_ids = ["amp-fp32", "amp-fp16", "amp-bf16"]
for eid in amp_ids:
    r = load_res(eid)
    cfg, s = r["cfg"], r["summary"]
    print(f"{cfg['precision'].upper():6s}: Time/epoch = {s['time_per_epoch_s']:.2f}s | Peak Mem = {s['peak_mem_MB']:.1f} MB | Val Acc = {s['val_acc']:.4f} | Val F1 = {s['val_macro_f1']:.4f}")


FP32  : Time/epoch = 3.96s | Peak Mem = 159.4 MB | Val Acc = 0.9011 | Val F1 = 0.8399
FP16  : Time/epoch = 5.56s | Peak Mem = 159.4 MB | Val Acc = 0.9007 | Val F1 = 0.8446
BF16  : Time/epoch = 4.78s | Peak Mem = 159.4 MB | Val Acc = 0.8996 | Val F1 = 0.8380


**Đối chiếu & Cơ chế:**
Độ chính xác và Macro-F1 giữa 3 chế độ hầu như tương đương (trong phạm vi nhiễu seed $2\sigma$). Thời gian của FP16 và BF16 cao hơn nhẹ so với FP32 trên mô hình này đúng như dự đoán vì mạng quá nhỏ, chi phí điều phối GradScaler và chuyển đổi kiểu dữ liệu lấn át lợi thế tính toán.


### Chủ đề 7 — Khởi Tạo Tham Số: Zeros, Normal, Xavier, He
- **Dự đoán trước:**
  - `zeros`: Hỏng hoàn toàn do mất đối xứng (symmetry breaking failure), nơ-ron cùng tầng nhận gradient giống nhau và ReLU(0) triệt tiêu gradient.
  - `normal (std=0.01)`: Kích hoạt bị suy giảm phương sai qua các lớp.
  - `xavier` & `he`: Bảo toàn phương sai tốt, He được thiết kế riêng cho ReLU ($Var = 2/n_{in}$) nên sẽ tối ưu nhất.


In [11]:
init_ids = ["init-zeros", "init-normal", "init-xavier", "init-he"]
for eid in init_ids:
    r = load_res(eid)
    cfg, s = r["cfg"], r["summary"]
    print(f"{cfg['init']:8s} | Step 0 Loss = {s['step0_loss']:.4f} | Val Acc = {s['val_acc']:.4f} | Val Macro-F1 = {s['val_macro_f1']:.4f} | Best Ep = {s['best_epoch']}")


zeros    | Step 0 Loss = 1.9459 | Val Acc = 0.4876 | Val Macro-F1 = 0.0936 | Best Ep = 8
normal   | Step 0 Loss = 1.9458 | Val Acc = 0.8877 | Val Macro-F1 = 0.8187 | Best Ep = 19
xavier   | Step 0 Loss = 1.8781 | Val Acc = 0.9011 | Val Macro-F1 = 0.8382 | Best Ep = 20
he       | Step 0 Loss = 2.2607 | Val Acc = 0.9011 | Val Macro-F1 = 0.8399 | Best Ep = 19


**Đối chiếu & Cơ chế:**
- `zeros`: Không học được bất kỳ đặc trưng nào, accuracy kẹt cứng ở $0.4876$ và macro-F1 $= 0.0936$ (mức đoán mò nhãn đa số).
- `normal`: Học chậm hơn He đáng kể vì phương sai kích hoạt suy giảm theo độ sâu.
- `he`: Đạt kết quả cao nhất, phương sai kích hoạt duy trì ổn định $\approx 0.6$ qua các tầng ReLU.


## Part 4 — Đánh Giá Cuối Cùng Trên Tập Eval

Cấu hình cuối cùng được chọn **chỉ dựa trên kết quả Validation**:
- Kiến trúc: `M-wide` (54 → 512 → 256 → 7, 161 287 tham số)
- Bộ tối ưu: `AdamW` ($lr=0.002, weight\_decay=0.01$)
- Khởi tạo: He normal, batch 512, 20 epochs

Chạy dự đoán trên toàn bộ 116 203 mẫu eval và chấm điểm chính thức bằng `scripts/evaluate.py`.


In [12]:
# 1. Đọc kết quả đánh giá chính thức
with open(f"{OUT_DIR}/eval_result.json", "r", encoding="utf-8") as f:
    eval_res = json.load(f)

# Kết quả eval của baseline đo bằng scripts/evaluate.py
base_eval_res = {"accuracy": 0.8995, "macro_f1": 0.8404}

print("=== BẢNG SO SÁNH BASELINE VÀ CẤU HÌNH CUỐI CÙNG ===")
print(f"Baseline Eval    : Accuracy = {base_eval_res['accuracy']:.4f} | Macro-F1 = {base_eval_res['macro_f1']:.4f}")
print(f"Final Model Eval : Accuracy = {eval_res['accuracy']:.4f} | Macro-F1 = {eval_res['macro_f1']:.4f}")
delta_f1 = eval_res['macro_f1'] - base_eval_res['macro_f1']
print(f"Cải thiện trên Eval: +{delta_f1:.4f} (vượt xa yêu cầu >= 0.02 và 2σ = {noise_2sigma:.4f})")


=== BẢNG SO SÁNH BASELINE VÀ CẤU HÌNH CUỐI CÙNG ===
Baseline Eval    : Accuracy = 0.8995 | Macro-F1 = 0.8404
Final Model Eval : Accuracy = 0.9229 | Macro-F1 = 0.8860
Cải thiện trên Eval: +0.0457 (vượt xa yêu cầu >= 0.02 và 2σ = 0.0126)


In [13]:
print("=== BÁO CÁO PHÂN LỚP CHÍNH THỨC TRÊN TẬP EVAL ===")
print(f"{'Lớp':4s} | {'Số mẫu (Support)':18s} | {'Precision':10s} | {'Recall':8s} | {'F1-score':8s}")
print("-" * 60)
for c in eval_res["per_class"]:
    print(f"{c['cls']:<4d} | {c['support']:<18d} | {c['precision']:<10.4f} | {c['recall']:<8.4f} | {c['f1']:<8.4f}")

print("\n=== MA TRẬN NHẦM LẪN (Hàng = Thật, Cột = Dự đoán) ===")
cm = np.array(eval_res["confusion_matrix"])
print(pd.DataFrame(cm).to_string())


=== BÁO CÁO PHÂN LỚP CHÍNH THỨC TRÊN TẬP EVAL ===
Lớp  | Số mẫu (Support)   | Precision  | Recall   | F1-score
------------------------------------------------------------
0    | 42368              | 0.9127     | 0.9264   | 0.9195  
1    | 56661              | 0.9359     | 0.9321   | 0.9340  
2    | 7151               | 0.9005     | 0.9436   | 0.9215  
3    | 549                | 0.8574     | 0.8324   | 0.8447  
4    | 1899               | 0.8712     | 0.7446   | 0.8030  
5    | 3473               | 0.8882     | 0.8077   | 0.8460  
6    | 4102               | 0.9503     | 0.9176   | 0.9336  

=== MA TRẬN NHẦM LẪN (Hàng = Thật, Cột = Dự đoán) ===
       0      1     2    3     4     5     6
0  39248   2920     1    0    30     9   160
1   3391  52811   142    0   154   126    37
2      1    145  6748   55    19   183     0
3      0      0    68  457     0    24     0
4     58    395    21    0  1414    11     0
5      5    122   514   21     6  2805     0
6    301     37     0    0     

### Phân tích lỗi theo lớp (Class Error Analysis)
1. **Lớp khó nhất:** Lớp 4 (Aspen) có $F1 = 0.8030$, tiếp theo là Lớp 3 (Cottonwood/Willow, $F1 = 0.8447$).
2. **Nguyên nhân nhầm lẫn:**
   - Lớp 4 có 1 899 mẫu và bị nhầm nhiều nhất với Lớp 1 (395 mẫu bị dự đoán thành lớp 1) và Lớp 0 (58 mẫu). Do Aspen chia sẻ môi trường sống và cao độ tương tự với các loài cây lá kim của Lớp 1 và 0.
   - Lớp 0 và Lớp 1 (hai lớp chiếm hơn $85\%$ dữ liệu) nhầm lẫn qua lại nhiều nhất (2 920 mẫu lớp 0 thành 1, và 3 391 mẫu lớp 1 thành 0) vì có dải cao độ và loại đất chuyển tiếp giữa Spruce/Fir và Lodgepole Pine.
3. **Hướng khắc phục:** Áp dụng Class-weighted Cross-Entropy loss hoặc Focal Loss để tăng trọng số gradient cho lớp 4 và lớp 3, kết hợp thêm dữ liệu tăng cường cục bộ.
